In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###Creating Women Texts Dataset

In [ ]:
woman_text_1="/content/drive/MyDrive/science_texts/cavendish.natpix.txt"
woman_text_2="/content/drive/MyDrive/science_texts/cavendish.observations.txt"
woman_text_3="/content/drive/MyDrive/science_texts/cavendish.philosophical.txt"
woman_text_4="/content/drive/MyDrive/science_texts/conway.philosophy.txt"
woman_text_5="/content/drive/MyDrive/science_texts/stone.midwifery.txt"
woman_text_6="/content/drive/MyDrive/science_texts/wolley.cook.txt"
test="/content/drive/MyDrive/science_texts/test/grey.secretphysick.txt"

woman_texts=[woman_text_1, woman_text_2, woman_text_3, woman_text_4, woman_text_5, woman_text_6, test]

In [ ]:
import nltk
nltk.download('punkt')
from nltk import tokenize

dict_sentence_w = {}

for i in woman_texts:
  file = open(i, "r")
  text = file.read()
  text = text[0:text.rfind("Finis.")]
  text = text[0:text.rfind("The End")]

  each_sentence = tokenize.sent_tokenize(text) #break text into sentences
  dict_sentence_w[i] = each_sentence

dict_ten_sentences_w = {}
for j in dict_sentence_w:
  dict_ten_sentences_w[j] = []
  for i in range(0, len(dict_sentence_w[j]), 10):
    ten_sentences = ""
    for k in range(10):
      if (i+k) < len(dict_sentence_w[j]):
        ten_sentences += dict_sentence_w[j][i+k] + " "
    dict_ten_sentences_w[j].append(ten_sentences.strip())

for j in dict_ten_sentences_w:
  dict_ten_sentences_w[j].pop(0)
  dict_ten_sentences_w[j].pop()
  dict_ten_sentences_w[j] = list(filter(lambda x: len(x)>=1000, dict_ten_sentences_w[j])) # remove long sentences


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


###Creating Men Texts Datset

In [ ]:
man_text_one = 'https://www.gutenberg.org/cache/epub/5500/pg5500.txt'
man_text_two = 'https://www.gutenberg.org/cache/epub/67065/pg67065.txt'
man_text_three = 'https://www.gutenberg.org/files/15491/15491-0.txt'
man_text_four = 'https://www.gutenberg.org/cache/epub/25830/pg25830.txt'
man_text_five = 'https://www.gutenberg.org/files/64097/64097-0.txt'
man_texts = [man_text_one, man_text_two, man_text_three, man_text_four, man_text_five]

from urllib import request
for url in man_texts:
  response = request.urlopen(url)
  text = response.read().decode('utf8')
  each_word = text.split()

dict_sentence_m = {}

for url in man_texts:
  response = request.urlopen(url)
  text = response.read().decode('utf8')
  a = text.find("*** START OF")
  if (a == -1):
    a = 0
  b = text.find("***START OF")
  if (b == -1):
    b = 0
  c = text.rfind("*** END OF")
  d = text.rfind("***END OF")
  text = text[a:c]
  text = text[b:d]
  text = text.replace("_", "")
  text = text.replace("\r\n", " ")
  each_sentence = tokenize.sent_tokenize(text) #break text into sentences
  dict_sentence_m[url] = each_sentence

dict_ten_sentences_m = {}
for j in dict_sentence_m:
  dict_ten_sentences_m[j] = []
  for i in range(0, len(dict_sentence_m[j]), 10):
    ten_sentences = ""
    for k in range(10):
      if (i+k) < len(dict_sentence_m[j]):
        ten_sentences += dict_sentence_m[j][i+k] + " "
    dict_ten_sentences_m[j].append(ten_sentences.strip())

for j in dict_ten_sentences_m:
  dict_ten_sentences_m[j].pop(0)
  dict_ten_sentences_m[j].pop()
  dict_ten_sentences_m[j] = list(filter(lambda x: len(x)>=1000, dict_ten_sentences_m[j]))

In [ ]:
w_training_data = []
test_data=[]
for i in dict_ten_sentences_w:
  if i != "/content/drive/MyDrive/science_texts/test/grey.secretphysick.txt":
    w_training_data.extend(dict_ten_sentences_w[i])
  else:
    test_data.extend(dict_ten_sentences_w[i])

m_training_data=[]
for i in dict_ten_sentences_m:
  m_training_data.extend(dict_ten_sentences_m[i])

###Features

In [ ]:
import pandas as pd
import spacy
nlp = spacy.load("en_core_web_sm")
featureWords=set() # all words that are pronouns, determiners, or punctuation marks
for txt in w_training_data+m_training_data+test_data:
  doc = nlp(txt)
  for word in doc:
    if word.pos_ == "PRON" or word.pos_ == "DET" or word.pos_ == "PUNCT":
      featureWords.add(str(word))


In [ ]:
features=[]
featureWords = list(featureWords)
for txt in w_training_data+m_training_data:
  feature=[0]*len(featureWords) # list that stores the number of appearences of each word in featureWords in the document
  words=txt.count(" ")+1
  for word in txt:
    if word in featureWords:
      i=featureWords.index(word)
      feature[i]+=1
  feature.append(words/10) # average words per sentence
  features.append(feature)

In [ ]:
features=pd.DataFrame(features, columns=featureWords+["words per sentence"])

In [ ]:
testFeatures=[]
for txt in test_data:
  feature=[0]*len(featureWords)
  words = txt.count(" ")+1
  for word in txt:
    if word in featureWords:
      i=featureWords.index(word)
      feature[i]+=1
  feature.append(words/10)
  testFeatures.append(feature)

testFeatures=pd.DataFrame(testFeatures, columns=featureWords+["words per sentence"])

In [ ]:
labels=pd.DataFrame([1]*len(w_training_data)+[0]*len(m_training_data))

###SVM Model

In [ ]:
from sklearn.model_selection import train_test_split
XTrain, XValidation, yTrain, yValidation = train_test_split(features, labels, test_size=0.2)

In [ ]:
from sklearn import svm
model=svm.SVC(kernel="linear", C=0.1)
model.fit(XTrain, yTrain)

/usr/local/lib/python3.7/dist-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVC(C=0.1, kernel='linear')

In [ ]:
from sklearn.metrics import accuracy_score
prediction=model.predict(XValidation)
accuracy=accuracy_score(yValidation, prediction)
print("Accuracy is "+str(round(accuracy*100, 2))+"%")

Accuracy is 95.86%


###Testing Model

In [ ]:
XTest=testFeatures.to_numpy()
datasize=XTest.shape[0]
yTest=[1]*datasize

prediction=model.predict(XTest)
accuracy=accuracy_score(yTest, prediction)
print("Accuracy is "+str(round(accuracy*100, 2))+"%")

Accuracy is 92.11%


/usr/local/lib/python3.7/dist-packages/sklearn/base.py:451: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  "X does not have valid feature names, but"
